# Real Estate Price Prediction

# Imports & Ingestion de données

In [ ]:
import modin.pandas as pd
from modin.config import AutoSwitchBackend; AutoSwitchBackend.disable()
import snowflake.snowpark.modin.plugin

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Baseline
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn import set_config
set_config(display = 'diagram')

# Sklearn preprocessing
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler

# Sklearn pipelines
from sklearn.compose import ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline, Pipeline

# Sklearn model selection
from scipy import stats
from sklearn.metrics import make_scorer, mean_squared_error, mean_squared_log_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, cross_validate

In [ ]:
# models
from sklearn.ensemble import AdaBoostRegressor,GradientBoostingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression, SGDRegressor, BayesianRidge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
USE DATABASE HOUSE_PRICE_DB;

CREATE SCHEMA IF NOT EXISTS HOUSE_SCHEMA;
USE SCHEMA HOUSE_SCHEMA;

In [ ]:
CREATE OR REPLACE FILE FORMAT JSONFORMAT 
    TYPE = 'JSON'
    STRIP_OUTER_ARRAY = TRUE;

CREATE STAGE IF NOT EXISTS HOUSE_ASSETS 
    FILE_FORMAT = JSONFORMAT 
    URL = 's3://logbrain-datalake/datasets/house_price/';

In [ ]:
LIST @HOUSE_ASSETS;

In [ ]:
SELECT $1
FROM @HOUSE_ASSETS
LIMIT 30;

In [ ]:
CREATE OR REPLACE TABLE HOUSE_PRICE_RAW (
    price INTEGER,
    area INTEGER,
    bedrooms INTEGER,
    bathrooms INTEGER,
    stories INTEGER,
    mainroad STRING,
    guestroom STRING,
    basement STRING,
    hotwaterheating STRING,
    airconditioning STRING,
    parking INTEGER,
    prefarea STRING,
    furnishingstatus STRING
)

In [ ]:
COPY INTO HOUSE_PRICE_RAW
FROM (
    SELECT 
        $1:price::INTEGER,
        $1:area::INTEGER,
        $1:bedrooms::INTEGER,
        $1:bathrooms::INTEGER,
        $1:stories::INTEGER,
        $1:mainroad::STRING,
        $1:guestroom::STRING,
        $1:basement::STRING,
        $1:hotwaterheating::STRING,
        $1:airconditioning::STRING,
        $1:parking::INTEGER,
        $1:prefarea::STRING,
        $1:furnishingstatus::STRING
    FROM @HOUSE_ASSETS
)
FILE_FORMAT = JSONFORMAT;

In [ ]:
SELECT * FROM HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_RAW
LIMIT 10

# Exploration de données

In [ ]:
df = pd.read_snowflake("SELECT * FROM HOUSE_PRICE_RAW")
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated().sum()/len(df)

In [ ]:
# Voir les doublons
print(f"Lignes totales : {len(df)}")
print(f"Doublons : {df.duplicated().sum()}")
print(f"Lignes uniques: {df.duplicated(keep=False) == False}.sum()")


df[df.duplicated(keep=False)].sort_values('PRICE').head(20)

Le dataset comporte beaucoup de doublons (environ 50%). Nous supprimons ces doublons afin d'éviter le data leakage et d'alourdir le modèle pour rien.

In [ ]:
df = df.drop_duplicates(ignore_index=True).to_pandas()
df.shape

## Distribution des prix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df.copy()

# Distribution de PRICE (variable cible)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(pdf['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution de PRICE")
axes[0].set_xlabel("Prix")
axes[0].set_ylabel("Fréquence")

# Boxplot pour détecter les outliers
axes[1].boxplot(pdf['PRICE'], vert=True)
axes[1].set_title("Boxplot PRICE")
axes[1].set_ylabel("Prix")

plt.tight_layout()
plt.show()

## Matrice de corrélation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df.copy()

# Encodage temporaire pour la corrélation
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 
               'AIRCONDITIONING', 'PREFAREA']
for c in binary_cols:
    pdf[c] = (pdf[c].str.lower() == 'yes').astype(int)

pdf['FURNISHINGSTATUS'] = pdf['FURNISHINGSTATUS'].map({
    'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0
})

# Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pdf.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation — House Price Dataset")
plt.tight_layout()
plt.show()

La plupart des features sont intéressantes pour notre modèle. Nous conservons Hotwaterheating (malgré sa corrélation à 9% avec notre target) pour l'entrainement du modèle car elle pourra être intéressante en feature engineering.

# X et y

In [ ]:
X = df.drop(columns=['PRICE'])
y = df['PRICE']

# Data splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

# Preprocessing

In [ ]:
X_preproc = X.copy()

In [ ]:
cols_binaires = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 
               'AIRCONDITIONING', 'PREFAREA']
for c in cols_binaires:
    X_preproc[c] = (X_preproc[c].str.lower() == 'yes').astype(int)

X_preproc['FURNISHINGSTATUS'] = X_preproc['FURNISHINGSTATUS'].map({
    'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0
})

In [ ]:
pdf = X_preproc.copy()
features = pdf.columns.tolist()

fig, axes = plt.subplots(len(features), 2, figsize=(14, 5 * len(features)))

for i, col in enumerate(features):
    # Histogramme
    axes[i, 0].hist(pdf[col], bins=30, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f"Distribution de {col}")
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel("Fréquence")

    # Boxplot
    axes[i, 1].boxplot(pdf[col], vert=True)
    axes[i, 1].set_title(f"Boxplot {col}")
    axes[i, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

# Pipeline

In [ ]:
FEATURE_COLS = list(X_train.columns)

In [ ]:
def preprocessing(X):
    import pandas as native_pd 
    if not isinstance(X, native_pd.DataFrame):
        X = native_pd.DataFrame(X, columns=FEATURE_COLS)
    X = X.copy()
    binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 
                   'AIRCONDITIONING', 'PREFAREA']
    for c in binary_cols:
        X[c] = (X[c].str.lower() == 'yes').astype(int)
    
    X['FURNISHINGSTATUS'] = X['FURNISHINGSTATUS'].map({
        'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0
    })
    return X

In [ ]:
def feature_engineering(X):
    import pandas as native_pd 
    if not isinstance(X, native_pd.DataFrame):
        X = native_pd.DataFrame(X, columns=FEATURE_COLS)
    X = X.copy()
    #Surface totale pondérée par étage
    X['AREA_PER_STORY']  = X['AREA'] / X['STORIES']
    # Ratio salles de bains par chambres
    X['BATH_BED_RATIO']  = X['BATHROOMS'] / X['BEDROOMS']

    # Score de confort du logement en fonction des avantages qu'il propose
    X['COMFORT_SCORE']   = (X['AIRCONDITIONING'] + X['HOTWATERHEATING'] + X['BASEMENT'] + X['GUESTROOM'])

    # Score de localisation du bien
    X['LOCATION_SCORE']  = X['MAINROAD'] + X['PREFAREA']

    # Parking oui ou non
    X['HAS_PARKING']     = (X['PARKING'] > 0).astype(int)

    # Score taille du bien et ameublement (plus un bien est grand et meublé plus il est premium)
    X['FURNISHED_AREA']  = X['AREA'] * X['FURNISHINGSTATUS']
    
    return X

Suite à l'analyse de la distribution précédente on note que Area possède des outliers. Nous appliquons donc un robust scaler sur les features concernées.

In [ ]:
area_cols    = ['AREA', 'AREA_PER_STORY', 'FURNISHED_AREA']
standard_cols = ['BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING',
                 'MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
                 'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS',
                 'BATH_BED_RATIO', 'COMFORT_SCORE', 'LOCATION_SCORE', 'HAS_PARKING']

In [ ]:
scaler = ColumnTransformer([
    ('robust',   RobustScaler(),   area_cols),
    ('standard', StandardScaler(), standard_cols)
])

In [ ]:
preproc = Pipeline([
    ('preprocessing',       FunctionTransformer(preprocessing)),
    ('feature_engineering', FunctionTransformer(feature_engineering)),
    ('scaler',              scaler)
])

In [ ]:
pipe_baseline = make_pipeline(preproc, 
                             LinearRegression())
pipe_baseline

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_baseline = cross_validate(pipe_baseline, X_train, y_train, 
                             scoring='r2', cv=kf)
print(cv_baseline['test_score'])
print(f"R² moyen : {cv_baseline['test_score'].mean():.4f}")

In [ ]:
pipe_baseline.fit(X_train, y_train)

In [ ]:
y_pred = pipe_baseline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=== Évaluation Train/Test ===")
print(f"MAE  : {mae:,.0f}")
print(f"RMSE : {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

# Test de plusieurs modèles différents

In [ ]:
final_scores = {
    'model_name': [],
    'score': [],
    'log': []
}

final_scores['model_name'].append('baseline')
final_scores['score'].append(cv_baseline['test_score'].mean())
#final_scores['log'].append(False)
final_scores

In [ ]:
models = [Ridge(), BayesianRidge(),
          SVR(),
          RandomForestRegressor(), AdaBoostRegressor(), GradientBoostingRegressor() ,
         XGBRegressor()
         ]

In [ ]:
final_scores = {'model_name': [], 'score': [], 'log': []}

for model in models:
    model_name = str(model)[:-2]
    print(f'Starting cv of model {model_name}')
    
    pipe = make_pipeline(preproc, model)
    cv_res = cross_validate(pipe, X_train, y_train, scoring='r2', cv=5)
    
    # Ajouter aux listes APRÈS le succès du cross_validate
    final_scores['model_name'].append(model_name)
    final_scores['score'].append(cv_res['test_score'].mean())
    final_scores['log'].append(False)

pd.DataFrame(final_scores).sort_values(by='score', ascending=False)

In [ ]:
# Évaluer les 3 candidats sur le test set
candidats = {
    'BayesianRidge': BayesianRidge(),
    'Ridge': Ridge(),
    'GradientBoosting': GradientBoostingRegressor()
}

for name, model in candidats.items():
    pipe = make_pipeline(preproc, model)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"{name:25s} R²={r2_score(y_test, y_pred):.4f}  MAE={mean_absolute_error(y_test, y_pred):,.0f}")

# Optimisation

## Optimisation GradientBoost

In [ ]:
param_grid_gb = {
    'gradientboostingregressor__n_estimators': [100, 200, 300],
    'gradientboostingregressor__max_depth': [2, 3, 4],
    'gradientboostingregressor__learning_rate': [0.05, 0.1, 0.2],
    'gradientboostingregressor__subsample': [0.7, 0.8, 1.0],
    'gradientboostingregressor__min_samples_leaf': [1, 3, 5]
}

pipe_gb = make_pipeline(preproc, GradientBoostingRegressor())

search_gb = RandomizedSearchCV(
    pipe_gb, param_grid_gb,
    n_iter=30, scoring='r2', cv=kf,
    random_state=42, n_jobs=1
)
search_gb.fit(X_train, y_train)

print(f"Best CV R²  : {search_gb.best_score_:.4f}")
print(f"Test R²     : {r2_score(y_test, search_gb.predict(X_test)):.4f}")
print(f"Best params : {search_gb.best_params_}")

## Optimisation BayesianRidge

In [ ]:
# BayesianRidge — moins de params mais vaut le coup
param_grid_br = {
    'bayesianridge__alpha_1': [1e-6, 1e-5, 1e-4],
    'bayesianridge__alpha_2': [1e-6, 1e-5, 1e-4],
    'bayesianridge__lambda_1': [1e-6, 1e-5, 1e-4],
    'bayesianridge__lambda_2': [1e-6, 1e-5, 1e-4],
}

pipe_br = make_pipeline(preproc, BayesianRidge())

search_br = RandomizedSearchCV(
    pipe_br, param_grid_br,
    n_iter=20, scoring='r2', cv=kf,
    random_state=42, n_jobs=1
)
search_br.fit(X_train, y_train)

print(f"Best CV R²  : {search_br.best_score_:.4f}")
print(f"Test R²     : {r2_score(y_test, search_br.predict(X_test)):.4f}")
print(f"Best params : {search_br.best_params_}")

Je pensais avoir un meilleur potentiel avec le GradientBoost mais le Bayesian Ridge est plus intéressant.

Au vu de l'écart entre le R² et le CV R² (crossvalidation), le GradientBoost tend à overfitter. Le dataset est surement trop petit.

### Test log

Pendant la phase d'exploration nous avons constaté que les prix possèdent beaucoup d'outliers

Pour optimiser nos résultats nous appliquons donc un logarithme sur notre target Price.

In [ ]:
from sklearn.compose import TransformedTargetRegressor


best_br_params = search_br.best_params_

model_log = TransformedTargetRegressor(
    regressor=BayesianRidge(**{
        k.replace('bayesianridge__', ''): v 
        for k, v in best_br_params.items()
    }),
    func=np.log1p,
    inverse_func=np.expm1
)

pipe_log_br = make_pipeline(preproc, model_log)

cv_log = cross_validate(pipe_log_br, X_train, y_train, scoring='r2', cv=kf)
pipe_log_br.fit(X_train, y_train)

print(f"R² CV   : {cv_log['test_score'].mean():.4f}")
print(f"R² test : {r2_score(y_test, pipe_log_br.predict(X_test)):.4f}")

# Evaluation du surapprentissage

In [ ]:
import altair as alt
alt.data_transformers.enable('json')
alt.theme.enable('opaque')

candidats = {
    'LinearRegression':    make_pipeline(preproc, LinearRegression()),
    'BayesianRidge':       make_pipeline(preproc, BayesianRidge()),
    'BayesianRidge+log':   pipe_log_br,
    'GradientBoosting':    make_pipeline(preproc, GradientBoostingRegressor()),
    'RandomForest':        make_pipeline(preproc, RandomForestRegressor()),
    'Ridge':               make_pipeline(preproc, Ridge()),
}

rows = []
for name, pipe in candidats.items():
    pipe.fit(X_train, y_train)
    r2_train = r2_score(y_train, pipe.predict(X_train))
    r2_test  = r2_score(y_test,  pipe.predict(X_test))
    rows.append({
        'Model':          name,
        'Training_R2':    round(r2_train, 4),
        'Testing_R2':     round(r2_test,  4),
        'Overfitting':    round(r2_train - r2_test, 4)
    })

import pandas as native_pd
overfitting_df = native_pd.DataFrame(rows).sort_values('Testing_R2', ascending=False)
print(overfitting_df.to_string(index=False))

Nous écartons automatiquement le RandomForest qui a un score d'overfitting élevé.

Le meilleur compromis performance/overfitting est le modèle BayesianRidge. Le modèle GradientBoosting est intéressant mais risqué en terme d'overfitting.

# Stacking

In [ ]:
estimators = [
    ('bayesian', BayesianRidge()),   
    ('ridge',    Ridge()),            
    ('gb',       GradientBoostingRegressor()),  # test car le R² est bon
]



stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=BayesianRidge(),  # le modèle avec le moins d'overfit
    cv=kf
)

pipe_stack = make_pipeline(preproc, stacking)
cv_stack = cross_validate(pipe_stack, X_train, y_train, scoring='r2', cv=kf)
pipe_stack.fit(X_train, y_train)

r2_train = r2_score(y_train, pipe_stack.predict(X_train))
r2_test  = r2_score(y_test,  pipe_stack.predict(X_test))



print(f"R² CV    : {cv_stack['test_score'].mean():.4f}")
print(f"R² train : {r2_train:.4f}")
print(f"R² test  : {r2_test:.4f}")
print(f"Overfit  : {r2_train - r2_test:.4f}")

# Voting

In [ ]:
from sklearn.ensemble import VotingRegressor

voting = VotingRegressor(estimators=[
    ('bayesian', BayesianRidge()),
    ('ridge',    Ridge()),
    ('gb',       GradientBoostingRegressor()),
])

pipe_vote = make_pipeline(preproc, voting)

cv_vote = cross_validate(pipe_vote, X_train, y_train, scoring='r2', cv=kf)
pipe_vote.fit(X_train, y_train)

r2_train = r2_score(y_train, pipe_vote.predict(X_train))
r2_test  = r2_score(y_test,  pipe_vote.predict(X_test))

print(f"R² CV    : {cv_vote['test_score'].mean():.4f}")
print(f"R² train : {r2_train:.4f}")
print(f"R² test  : {r2_test:.4f}")
print(f"Overfit  : {r2_train - r2_test:.4f}")

Le Voting bat le Stacking avec nos meilleurs modèles sur le R². L'overfitting reste correct. Nous conservons ce modèle pour la production.

# Registry

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="HOUSE_PRICE_DB", schema_name="HOUSE_SCHEMA")

reg.log_model(
    pipe_log_br,
    model_name="house_price_voting",
    version_name="v1",
    sample_input_data=X_train, 
    metrics={
        "r2_cv":   0.6585,
        "r2_test": 0.6895,
        "overfitting": 0.0828
    },
    comment="VotingRegressor (BayesianRidge + Ridge + GradientBoosting)"
)

# Inférence

Sauvegarde des données test dans une table

In [ ]:
test_df = X_test.copy()
test_df['ACTUAL_PRICE'] = y_test.values

snowpark_df = session.create_dataframe(test_df)
snowpark_df.write.mode("overwrite").save_as_table(
    "HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_TEST_DATA"
)

print(f"{len(test_df)} lignes sauvegardées dans HOUSE_PRICE_TEST_DATA")

Inférence avec SQL

In [ ]:
SELECT 
    AREA, 
    BEDROOMS, 
    BATHROOMS, 
    STORIES,
    MAINROAD, 
    GUESTROOM, 
    BASEMENT, 
    HOTWATERHEATING,
    AIRCONDITIONING, 
    PARKING, 
    PREFAREA, 
    FURNISHINGSTATUS,
    ACTUAL_PRICE,

    HOUSE_PRICE_VOTING!PREDICT(AREA, BEDROOMS, BATHROOMS, STORIES,MAINROAD, GUESTROOM, 
                               BASEMENT, HOTWATERHEATING,AIRCONDITIONING, PARKING, PREFAREA, 
                               FURNISHINGSTATUS
    ) AS PREDICTED_PRICE
FROM HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_TEST_DATA
LIMIT 10;

Nous devons parser les prédictons qui sortent en JSON actuellement

In [ ]:
CREATE OR REPLACE TABLE HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_PREDICTIONS AS
SELECT
    AREA, 
    BEDROOMS, 
    BATHROOMS, 
    STORIES,
    MAINROAD, 
    GUESTROOM, 
    BASEMENT, 
    HOTWATERHEATING,
    AIRCONDITIONING, 
    PARKING, 
    PREFAREA, 
    FURNISHINGSTATUS,
    ACTUAL_PRICE,
    
    HOUSE_PRICE_VOTING!PREDICT(
        AREA, BEDROOMS, BATHROOMS, STORIES,
        MAINROAD, GUESTROOM, BASEMENT, HOTWATERHEATING,
        AIRCONDITIONING, PARKING, PREFAREA, FURNISHINGSTATUS
    )['output_feature_0']::FLOAT AS PREDICTED_PRICE
    
FROM HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_TEST_DATA;

Vérification des résultats

In [ ]:
SELECT 
    ACTUAL_PRICE,
    PREDICTED_PRICE,
    ABS(ACTUAL_PRICE - PREDICTED_PRICE) AS ERROR_ABS
    
FROM HOUSE_PRICE_DB.HOUSE_SCHEMA.HOUSE_PRICE_PREDICTIONS
ORDER BY ERROR_ABS

LIMIT 10;